In [2]:
import pandas as pd 
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor

In [6]:
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=5,
    n_redundant=10,
    flip_y=0.2,
    random_state=42
)

df = pd.DataFrame(
    X,
    columns=[f'feature_{i}' for i in range(20)]
)

df['target'] = y

train_data, test_data = train_test_split(
    df,
    test_size=0.3,
    random_state=42
)

print(train_data)
print(f"Training samples: {len(train_data)} | Testing samples: {len(test_data)}")
print(f"Training samples: {len(train_data)} | Testing samples: {len(test_data)}\n")

     feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
541   0.714389   0.068249   0.957234   1.496675   2.244351  -0.012228   
440   3.209313   1.686740   0.341922  -0.899848  -0.601669  -0.232910   
482  -1.320132  -1.048478  -1.890106   1.580536   0.493156   0.024240   
422   2.449954   2.948900   0.184840   1.437898  -1.786023  -1.773719   
778   5.252559   0.505531  -0.194180  -2.030131   4.161342   1.078719   
..         ...        ...        ...        ...        ...        ...   
106   1.740627   3.469629  -0.699510   1.525312  -6.071166   0.295237   
270   0.464653   0.228400   1.160402   2.560363   1.430410  -1.103251   
860   0.954256  -1.359125   1.997826  -1.292187   3.148682   0.869128   
435   1.028183   1.841429   0.312387   1.852995  -2.042713  -1.063283   
102   1.292783   1.159451  -1.869927   0.484071   0.909652  -1.188597   

     feature_6  feature_7  feature_8  feature_9  ...  feature_11  feature_12  \
541  -1.850611   1.445106   0.389342   2.09

In [8]:
print("--- Training Underfit Model ---")

underfit_predictor = TabularPredictor(
    label='target',
    verbosity=0
).fit(
    train_data,
    hyperparameters={
        'RF': {
            'max_depth': 1,
            'n_estimators': 5
        }
    },
    num_bag_folds=0
)

--- Training Underfit Model ---


In [10]:
print("--- Training Overfit Model ---")

overfit_predictor = TabularPredictor(
    label='target',
    verbosity=0
).fit(
    train_data,
    hyperparameters={
        'GBM': {
            'num_leaves': 999,
            'min_data_in_leaf': 1,
            'learning_rate': 0.5,
            'num_boost_round': 500
        }
    },
    num_bag_folds=0
)

--- Training Overfit Model ---


In [11]:
print("--- Training Optimal Model (AutoGluon Defaults) ---")
optimal_predictor = TabularPredictor(label = 'target', verbosity = 0).fit(
    train_data,
    presets = 'best_quality',
    time_limit = 60
)

--- Training Optimal Model (AutoGluon Defaults) ---


c:\Users\NishigandhaChoudhury\AppData\Local\Programs\Python\Python313\Lib\site-packages\autogluon\tabular\predictor\predictor.py:1705: UserWarning: Failed to use ray for memory safe fits. Falling back to normal fit. Error: ImportError('ray is required to train folds in parallel for TabularPredictor or HPO for MultiModalPredictor. A quick tip is to install via `pip install "ray>=2.43.0,<2.57.0"`')
  stacked_overfitting = self._sub_fit_memory_save_wrapper(


In [13]:
def evaluate_model(name, predictor):
    train_acc = predictor.evaluate(train_data, silent=True)['accuracy']
    test_acc = predictor.evaluate(test_data, silent=True)['accuracy']
    gap = train_acc - test_acc

    print(
        f"{name.ljust(15)} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Acc: {test_acc:.4f} | "
        f"Gap: {gap:.4f}"
    )


print("\n=== RESULTS: THE GENERALIZATION GAP ===")

evaluate_model("Underfitting", underfit_predictor)
evaluate_model("Overfitting", overfit_predictor)
evaluate_model("Optimal Fit", optimal_predictor)


=== RESULTS: THE GENERALIZATION GAP ===
Underfitting    | Train Acc: 0.7429 | Test Acc: 0.7500 | Gap: -0.0071
Overfitting     | Train Acc: 0.9514 | Test Acc: 0.7200 | Gap: 0.2314
Optimal Fit     | Train Acc: 0.9757 | Test Acc: 0.8367 | Gap: 0.1390
